In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

In [3]:
products = pd.read_excel("transformed_data/products_transformed.xlsx")

In [4]:
products.head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,PROD00001,fashion,57,1772,7,5088.91,56.00,4.71,32.200
1,PROD00002,home_decor,75,1944,2,7464.63,55.79,48.93,75.960
2,PROD00003,pet_supplies,24,1829,4,5749.25,27.38,34.04,43.170
3,PROD00004,kitchen,48,1404,6,3417.74,5.79,50.34,11.210
4,PROD00005,fashion,13,1350,1,1642.05,24.13,20.10,41.805


In [5]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

### create combined features

In [7]:
products["combined_features"] = (
    products["product_category_name"].astype(str)
)

### view features

In [9]:
products[[
    "product_id",
    "combined_features"
]].head()

,product_id,combined_features
0,PROD00001,fashion
1,PROD00002,home_decor
2,PROD00003,pet_supplies
3,PROD00004,kitchen
4,PROD00005,fashion


### create TF-IDF matrix

In [11]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    products["combined_features"]
)

### check matrix shape

In [13]:
tfidf_matrix.shape

(4847, 13)

### create cosine similarity matrix

In [15]:
similarity_matrix = cosine_similarity(
    tfidf_matrix
)

### check similarity matrix

In [17]:
similarity_matrix.shape

(4847, 4847)

### create product index mapping

In [19]:
product_index = pd.Series(
    products.index,
    index=products["product_id"]
).drop_duplicates()

### create recommendation function

In [21]:
def recommend_products(product_id, top_n=5):

    if product_id not in product_index:
        print("Product not found")
        return

    idx = product_index[product_id]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    product_indices = [
        i[0]
        for i in similarity_scores
    ]

    recommendations = products.iloc[
        product_indices
    ][[
        "product_id",
        "product_category_name"
    ]]

    return recommendations

### test function

In [23]:
recommend_products("PROD00001")

,product_id,product_category_name
4,PROD00005,fashion
12,PROD00013,fashion
19,PROD00020,fashion
74,PROD00075,fashion
88,PROD00089,fashion


In [ ]:
recommend_products("")